# CV Masterclass 09: Video & 3D Vision

Welcome to Notebook 09. A photograph is a 2D projection of a 3D reality, frozen at a specific Time $T$. 

When we feed raw video into a Neural Network, we are adding the $T$ dimension. When we try to reconstruct spatial depth, we are adding the $Z$ dimension. Both of these cause catastrophic parameter explosions if handled naively.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"Processing a 5-second 60FPS video through a standard 3D CNN requires more memory than an Nvidia H100 GPU physically possesses. How do modern video architectures geometrically decouple space and time to survive this?"*

---

## Table of Contents
1. **3D Convolutions:** The parameter explosion of Spatiotemporal volumes.
2. **(2+1)D Factorization:** Decoupling space and time.
3. **Optical Flow:** The physics of motion vectors.
4. **Neural Radiance Fields (NeRFs):** Ray casting and 3D synthesis.


## 1. 3D Convolutions & The Parameter Explosion

A standard image convolution kernel is $(K_H, K_W)$. For example, $3\times3=9$ parameters.

A video is a sequence of images $(T, H, W)$. To capture motion, we inflate the kernel into a 3D block: $(K_T, K_H, K_W)$. 
A $3\times3\times3$ kernel has $27$ parameters. 

If a Layer has 512 input channels and 512 output channels:
*   **2D Layer Params:** $3\times3 \times 512 \times 512 = 2.3M$
*   **3D Layer Params:** $3\times3\times3 \times 512 \times 512 = 7.0M$

The parameter count triples. The memory required to store the intermediate activation maps across 300 frames of video explodes into the Terabytes. You cannot train a 100-layer 3D ResNet natively.

## 2. (2+1)D Factorization

### 🎓 DEEP QUESTION ANSWERED
**Q:** *How do modern architectures geometrically decouple space and time to survive the memory explosion?*

**A:** 
They factorize the 3D convolution into a 2D spatial convolution strictly followed by a 1D temporal convolution. This is called the **(2+1)D architecture**.

Instead of a single $3\times3\times3$ step (cost: 27 ops):
1. **Spatial Step:** Apply a $1\times3\times3$ kernel (Looks at 1 frame, analyzes $3\times3$ space). Cost: 9 ops.
2. **Temporal Step:** Apply a $3\times1\times1$ kernel (Looks at 1 pixel across 3 frames of time). Cost: 3 ops.

**Total Cost:** $9 + 3 = 12$ ops.
By decoupling space and time, the cost drops by more than $50\%$, while adding an extra ReLU activation between the two steps, making the network actually *more* non-linear and expressive than the native 3D tensor!

## 3. Optical Flow

Even with (2+1)D factorization, feeding raw pixels into a network is inefficient for recognizing *action* (e.g., "Running" vs "Walking").

Most of the pixels in a video (the background) never move. 
**Optical Flow** algorithms (like Lucas-Kanade) calculate the physical $(dx, dy)$ velocity vector of every pixel between Frame $t$ and Frame $t+1$.

By feeding the network two streams—Stream A (Raw Pixels for *What* it is) and Stream B (Optical Flow Vectors for *How* it is moving)—the network converges drastically faster. The Optical Flow pre-computes the physics of motion so the CNN doesn't have to.

## 4. Neural Radiance Fields (NeRFs)

How do we generate a full 3D environment from 20 basic photographs taken from different angles?

A NeRF is an MLP (Dense Neural Network). It does not output an image. 
It takes exactly two inputs: $(x,y,z)$ coordinate in a 3D room, and $(\theta, \phi)$ viewing angle of the camera.
It outputs exactly two things: The RGB Color of that point, and the Density (is it solid or transparent?).

During inference, we cast thousands of mathematical 'Rays' through a conceptual camera grid into the virtual room. For every point along the ray, we query the MLP for color and density, and integrate them using classical ray-tracing physics to render the final 2D image pixel. 

Because the MLP is continuous, it has infinite resolution. You can literally zoom in forever.